# Ask-SLM Runner (Interactive)


In [1]:
import sys
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "Managers").exists() else cwd.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

try:
    display  # type: ignore[name-defined]
except NameError:
    def display(obj):
        print(obj)

from managers.analysis_pipeline_manager import AnalysisPipelineManager
from managers.mapping_config_manager import MappingConfigManager


In [2]:
workspace = "DemoStudy"
manager = AnalysisPipelineManager()

workspace_root = project_root / "Workspaces"

init_result = manager.initialize_workspace(
    workspace=workspace,
    workspace_root=workspace_root,
)
context = init_result.context
runner_logger = init_result.logger
print(f"Using workspace root: {workspace_root}")


[2026-05-24 11:00:39] Log file path: /Users/yarintamam/Code/Ask-SLM/Workspaces/DemoStudy/Logs/run_20260524_110039_463593.log
Using workspace root: /Users/yarintamam/Code/Ask-SLM/Workspaces


In [3]:
inputs_result = manager.prepare_workspace_inputs(
    context=context,
    logger=runner_logger,
)
metadata_table = inputs_result.metadata_table
taxonomy_table = inputs_result.taxonomy_table

mapping_result = manager.refresh_mapping_templates(
    metadata_df=metadata_table,
    context=context,
    logger=runner_logger,
)
sra_processor = mapping_result.processor
project_ids = mapping_result.project_ids

display(mapping_result.as_dataframe())


,project_id,config_exists,config_path,template_path
0,PRJNA974858,False,/Users/yarintamam/Code/Ask-SLM/Workspaces/Demo...,/Users/yarintamam/Code/Ask-SLM/Workspaces/Demo...


In [4]:
# Keep mapping review explicit in the notebook flow.
MappingConfigManager(
    config_dir=context.config_dir,
    logger=runner_logger.log,
).run()


✅ Loaded 27 fields from Metalog schema
📋 Loaded 1 project(s)



In [ ]:
enrichment_result = manager.run_enrichment_stage(
    metadata_df=metadata_table,
    processor=sra_processor,
    project_ids=project_ids,
    context=context,
    logger=runner_logger,
    assembly_db_dir=context.cache_dir,
    strict_missing_configs=False,
)
enriched_metadata = enrichment_result.enriched_metadata

envo_result = manager.enrich_envo_columns(
    metadata_df=enriched_metadata,
    context=context,
    logger=runner_logger,
)
enriched_metadata = envo_result.enriched_metadata

display(
    pd.DataFrame(
        {
            "metric": ["samples", "projects", "mapped_columns", "missing_mapping_projects", "envo_terms_resolved"],
            "value": [
                len(enriched_metadata),
                enriched_metadata["bioproject"].nunique(),
                len(enrichment_result.mapped_cols),
                len(enrichment_result.missing_mapping_projects),
                len(envo_result.envo_report_df),
            ],
        }
    )
)


In [ ]:
taxonomy_result = manager.run_taxonomy_stage(
    taxonomy_df=taxonomy_table,
    context=context,
    logger=runner_logger,
    assembly_db_dir=context.cache_dir,
    merged_dump_path=project_root / "Accessors/TaxonomyAccessor/merged.dmp",
    taxon_key_sep="||",
)

merge_result = manager.run_merge_stage(
    enriched_metadata_df=enriched_metadata,
    abundance_matrix=taxonomy_result.abundance_matrix,
    context=context,
    logger=runner_logger,
    assembly_db_dir=context.cache_dir,
)

abundance_metadata_matrix = merge_result.abundance_metadata_matrix
metadata_columns = merge_result.metadata_columns
taxon_lookup_df = taxonomy_result.taxon_lookup_df

display(abundance_metadata_matrix[metadata_columns].head())


In [ ]:
shared_tax_accessor = manager.build_or_load_tax_accessor(
    taxon_lookup_df=taxon_lookup_df,
    context=context,
    logger=runner_logger,
    assembly_db_dir=context.cache_dir,
    lineage_db_path=context.cache_dir / "taxonomy_lineage_db.json",
)


In [ ]:
manager.run_alpha_diversity(
    analysis_wide_df=abundance_metadata_matrix,
    taxon_lookup_df=taxon_lookup_df,
    metadata_cols=metadata_columns,
    tax_accessor=shared_tax_accessor,
    logger=runner_logger,
)


In [ ]:
manager.run_taxonomic_abundance(
    abundance_metadata_matrix=abundance_metadata_matrix,
    taxon_lookup_df=taxon_lookup_df,
    metadata_cols=metadata_columns,
    logger=runner_logger,
)
